# A Comparative Study of Optimization Algorithms for CNN-Based Image Classification
**AI Deep Learning Lab – Milestone 2** | Track: Category 1 (Experimentation-Based)
**Author:** Vataliya Mitesh Kiritbhai (AU2644030), M.Tech CSE

This notebook trains the *same* CNN on CIFAR-10 with **SGD, Adam, RMSprop and AdamW**, repeats each run over several seeds, and exports every table, figure and LaTeX macro used by the report and slides into `results/`.

**How to run (Google Colab):** Runtime → Change runtime type → **GPU (T4)** → Runtime → **Run all**. Expect roughly 45-90 minutes for the full protocol. Progress is checkpointed to `results/raw_results.json`, so if Colab disconnects, re-run and finished runs are skipped.

## 1. Setup

In [ ]:
import os, json, time, random, math
import numpy as np
import pandas as pd
import torch, torch.nn as nn
import torchvision, torchvision.transforms as T
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix
import matplotlib
import matplotlib.pyplot as plt

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| torchvision", torchvision.__version__, "| device:", DEVICE)
if DEVICE.type == "cpu":
    print("WARNING: no GPU detected. Enable it via Runtime > Change runtime type > GPU.")

OUT = "results"
os.makedirs(f"{OUT}/figures", exist_ok=True)
RAW = f"{OUT}/raw_results.json"

## 2. Experiment configuration
Everything except the optimizer is held fixed (data, split, augmentation, architecture, initial weights per seed, loss, batch size, epochs, evaluation).
Learning rates cannot be shared across optimizers (SGD and adaptive methods operate on very different scales), so each optimizer uses a standard default; Section 9 checks sensitivity to this choice.

In [ ]:
QUICK = False          # True = smoke test (2 epochs, 1 seed, no sweep) to check the pipeline in ~2 minutes
RUN_LR_SWEEP = True    # learning-rate sensitivity ablation (Section 9)

CFG = dict(
    epochs      = 2 if QUICK else 30,
    batch_size  = 128,
    seeds       = [0] if QUICK else [0, 1, 2],
    val_size    = 5000,
    split_seed  = 42,                 # fixed train/val split for ALL runs
    num_workers = 2,
    conv_thresh = 0.75,               # "epochs to reach 75% validation accuracy"
    sweep_epochs= 2 if QUICK else 15,
    sweep_seed  = 0,
)
OPTIMIZERS = {   # order matters (tables/plots)
    "SGD":     dict(cls=torch.optim.SGD,     kwargs=dict(lr=0.01, momentum=0.9)),
    "Adam":    dict(cls=torch.optim.Adam,    kwargs=dict(lr=1e-3)),
    "RMSprop": dict(cls=torch.optim.RMSprop, kwargs=dict(lr=1e-3)),
    "AdamW":   dict(cls=torch.optim.AdamW,   kwargs=dict(lr=1e-3, weight_decay=1e-2)),
}
SWEEP_LRS = {
    "SGD":     [0.1, 0.01, 0.001],
    "Adam":    [1e-2, 1e-3, 1e-4],
    "RMSprop": [1e-2, 1e-3, 1e-4],
    "AdamW":   [1e-2, 1e-3, 1e-4],
}
CLASSES = ["airplane","automobile","bird","cat","deer","dog","frog","horse","ship","truck"]
print(json.dumps({k:v for k,v in CFG.items()}, indent=1))
for k,v in OPTIMIZERS.items(): print(k, v["kwargs"])

## 3. Data: CIFAR-10 (45,000 train / 5,000 validation / 10,000 test)

In [ ]:
MEAN, STD = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)
train_tf = T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip(), T.ToTensor(), T.Normalize(MEAN, STD)])
eval_tf  = T.Compose([T.ToTensor(), T.Normalize(MEAN, STD)])

ds_train_aug  = torchvision.datasets.CIFAR10("./data", train=True,  download=True, transform=train_tf)
ds_train_eval = torchvision.datasets.CIFAR10("./data", train=True,  download=False, transform=eval_tf)
ds_test       = torchvision.datasets.CIFAR10("./data", train=False, download=True, transform=eval_tf)

perm = torch.randperm(len(ds_train_aug), generator=torch.Generator().manual_seed(CFG["split_seed"])).tolist()
val_idx, tr_idx = perm[:CFG["val_size"]], perm[CFG["val_size"]:]
train_set = Subset(ds_train_aug,  tr_idx)     # augmented
val_set   = Subset(ds_train_eval, val_idx)    # clean
print(len(train_set), len(val_set), len(ds_test))

def make_loaders(seed):
    g = torch.Generator().manual_seed(seed)
    kw = dict(batch_size=CFG["batch_size"], num_workers=CFG["num_workers"], pin_memory=(DEVICE.type=="cuda"))
    return (DataLoader(train_set, shuffle=True, generator=g, **kw),
            DataLoader(val_set, shuffle=False, **kw),
            DataLoader(ds_test, shuffle=False, **kw))

## 4. Model: the same custom CNN for every experiment

In [ ]:
class CNN(nn.Module):
    """3 conv blocks (2x [Conv3x3-ReLU] + MaxPool) -> FC(256) -> Dropout -> FC(10)."""
    def __init__(self, num_classes=10, p=0.3):
        super().__init__()
        def block(i, o):
            return nn.Sequential(nn.Conv2d(i,o,3,padding=1), nn.ReLU(inplace=True),
                                 nn.Conv2d(o,o,3,padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2))
        self.features = nn.Sequential(block(3,32), block(32,64), block(64,128))
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(128*4*4, 256), nn.ReLU(inplace=True),
                                        nn.Dropout(p), nn.Linear(256, num_classes))
    def forward(self, x): return self.classifier(self.features(x))

N_PARAMS = sum(p.numel() for p in CNN().parameters())
print(f"Trainable parameters: {N_PARAMS:,}")   # 814,122

## 5. Training & evaluation utilities

In [ ]:
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def sync():
    if DEVICE.type == "cuda": torch.cuda.synchronize()

@torch.no_grad()
def evaluate(model, loader, crit):
    model.eval(); tot_loss = 0.0; preds, labels = [], []
    for x, y in loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        out = model(x)
        tot_loss += crit(out, y).item() * x.size(0)
        preds.append(out.argmax(1).cpu()); labels.append(y.cpu())
    preds, labels = torch.cat(preds).numpy(), torch.cat(labels).numpy()
    return tot_loss / len(labels), float((preds == labels).mean()), preds, labels

def train_run(opt_name, seed, epochs, lr=None):
    """Train one model; select the epoch with best validation accuracy; report on the test set."""
    set_seed(seed)
    train_loader, val_loader, test_loader = make_loaders(seed)
    model = CNN().to(DEVICE)                       # same init for every optimizer given a seed
    spec = OPTIMIZERS[opt_name]; kw = dict(spec["kwargs"])
    if lr is not None: kw["lr"] = lr
    opt = spec["cls"](model.parameters(), **kw)
    crit = nn.CrossEntropyLoss()
    h = dict(train_loss=[], train_acc=[], val_loss=[], val_acc=[], epoch_time=[])
    best_val, best_state, best_ep = -1.0, None, -1
    for ep in range(epochs):
        model.train(); sync(); t0 = time.time()
        run_loss, correct, n = 0.0, 0, 0
        for x, y in train_loader:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            out = model(x); loss = crit(out, y)
            loss.backward(); opt.step()
            run_loss += loss.item() * x.size(0); correct += (out.argmax(1) == y).sum().item(); n += x.size(0)
        sync(); dt = time.time() - t0              # training time only (includes data loading)
        vl, va, _, _ = evaluate(model, val_loader, crit)
        h["train_loss"].append(run_loss / n); h["train_acc"].append(correct / n)
        h["val_loss"].append(vl); h["val_acc"].append(va); h["epoch_time"].append(dt)
        if va > best_val:
            best_val, best_ep = va, ep
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        print(f"  [{opt_name} s{seed} lr={kw['lr']:g}] ep {ep+1:02d}/{epochs} "
              f"loss {h['train_loss'][-1]:.3f} | val acc {va:.4f} | {dt:.1f}s", flush=True)
    model.load_state_dict(best_state)
    tl, ta, preds, labels = evaluate(model, test_loader, crit)
    p, r, f, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
    thr = next((i + 1 for i, v in enumerate(h["val_acc"]) if v >= CFG["conv_thresh"]), None)
    return dict(optimizer=opt_name, seed=seed, lr=kw["lr"], epochs=epochs, hist=h,
                test_loss=tl, test_acc=ta * 100, precision=p * 100, recall=r * 100, f1=f * 100,
                best_epoch=best_ep + 1, best_val_acc=best_val * 100, epochs_to_thr=thr,
                train_time=float(sum(h["epoch_time"])), cm=confusion_matrix(labels, preds).tolist())

# checkpointed store (resume after Colab disconnects)
store = json.load(open(RAW)) if os.path.exists(RAW) else {}
def run_cached(key, **kw):
    if key in store: print("cached:", key); return store[key]
    store[key] = train_run(**kw)
    json.dump(store, open(RAW, "w"))
    return store[key]

## 6. Main experiment: 4 optimizers × seeds

In [ ]:
for seed in CFG["seeds"]:
    for name in OPTIMIZERS:
        run_cached(f"main|{name}|{seed}", opt_name=name, seed=seed, epochs=CFG["epochs"])
print("Main experiment done.")

## 7. Aggregate results

In [ ]:
rows = [v for k, v in store.items() if k.startswith("main|") and v["epochs"] == CFG["epochs"]]
df = pd.DataFrame([{k: v for k, v in r.items() if k not in ("hist", "cm")} for r in rows])
metrics = ["test_acc", "precision", "recall", "f1", "test_loss", "train_time", "best_epoch", "epochs_to_thr"]
g = df.groupby("optimizer")[metrics]
summary = pd.concat({"mean": g.mean(), "std": g.std().fillna(0)}, axis=1).reindex(list(OPTIMIZERS))
summary.columns = [f"{b}_{a}" for a, b in summary.columns]
summary["sec_per_epoch_mean"] = summary["train_time_mean"] / CFG["epochs"]
summary.to_csv(f"{OUT}/summary.csv"); df.to_csv(f"{OUT}/all_runs.csv", index=False)
summary.round(3)

## 8. Figures

In [ ]:
plt.rcParams.update({"figure.dpi": 130, "font.size": 9, "axes.grid": True, "grid.alpha": .3})
COL = dict(zip(OPTIMIZERS, ["#1f77b4", "#d62728", "#2ca02c", "#9467bd"]))

def stack(opt, key):
    return np.array([r["hist"][key] for r in rows if r["optimizer"] == opt])

def curve_fig(keys, titles, ylabels, fname, sharey=False):
    fig, axs = plt.subplots(1, len(keys), figsize=(3.6 * len(keys), 2.9), sharey=sharey)
    axs = np.atleast_1d(axs)
    for ax, key, ttl, yl in zip(axs, keys, titles, ylabels):
        for o in OPTIMIZERS:
            a = stack(o, key); m, s = a.mean(0), a.std(0); x = np.arange(1, len(m) + 1)
            ax.plot(x, m, label=o, color=COL[o]); ax.fill_between(x, m - s, m + s, color=COL[o], alpha=.15)
        ax.set_title(ttl); ax.set_xlabel("Epoch"); ax.set_ylabel(yl)
    axs[0].legend(frameon=False)
    plt.tight_layout()
    for ext in ("pdf", "png"): plt.savefig(f"{OUT}/figures/{fname}.{ext}", bbox_inches="tight")
    plt.show()

curve_fig(["train_loss", "val_loss"], ["Training loss", "Validation loss"], ["Cross-entropy"] * 2, "loss_curves")
curve_fig(["val_acc"], ["Validation accuracy"], ["Accuracy"], "val_acc_curves")

# bar charts: accuracy + time
fig, axs = plt.subplots(1, 2, figsize=(7.2, 2.9))
x = np.arange(len(OPTIMIZERS)); names = list(OPTIMIZERS)
axs[0].bar(x, summary["test_acc_mean"], yerr=summary["test_acc_std"], capsize=3, color=[COL[o] for o in names])
lo = summary["test_acc_mean"].min(); axs[0].set_ylim(max(0, lo - 8), min(100, summary["test_acc_mean"].max() + 3))
axs[0].set_title("Test accuracy (%)"); axs[0].set_xticks(x); axs[0].set_xticklabels(names)
axs[1].bar(x, summary["sec_per_epoch_mean"], color=[COL[o] for o in names])
axs[1].set_title("Training time per epoch (s)"); axs[1].set_xticks(x); axs[1].set_xticklabels(names)
plt.tight_layout()
for ext in ("pdf", "png"): plt.savefig(f"{OUT}/figures/accuracy_time_bars.{ext}", bbox_inches="tight")
plt.show()

# confusion matrix of best optimizer (summed over seeds)
best_opt = summary["test_acc_mean"].idxmax()
cm = sum(np.array(r["cm"]) for r in rows if r["optimizer"] == best_opt)
cmn = cm / cm.sum(1, keepdims=True)
fig, ax = plt.subplots(figsize=(4.6, 4.1))
im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1); ax.grid(False)
ax.set_xticks(range(10)); ax.set_yticks(range(10))
ax.set_xticklabels(CLASSES, rotation=60, ha="right"); ax.set_yticklabels(CLASSES)
for i in range(10):
    for j in range(10):
        ax.text(j, i, f"{cmn[i, j]:.2f}", ha="center", va="center", fontsize=6, color="white" if cmn[i, j] > .5 else "black")
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(f"Normalised confusion matrix ({best_opt})")
plt.tight_layout()
for ext in ("pdf", "png"): plt.savefig(f"{OUT}/figures/confusion_matrix.{ext}", bbox_inches="tight")
plt.show()

## 9. Ablation: learning-rate sensitivity (single seed, fewer epochs)

In [ ]:
if RUN_LR_SWEEP:
    for name, lrs in SWEEP_LRS.items():
        for lr in lrs:
            run_cached(f"sweep|{name}|{lr:g}", opt_name=name, seed=CFG["sweep_seed"], epochs=CFG["sweep_epochs"], lr=lr)
sw = [v for k, v in store.items() if k.startswith("sweep|") and v["epochs"] == CFG["sweep_epochs"]]
sweep_df = pd.DataFrame([{k: v for k, v in r.items() if k not in ("hist", "cm")} for r in sw])
if len(sweep_df):
    sweep_df["optimizer"] = pd.Categorical(sweep_df["optimizer"], list(OPTIMIZERS), ordered=True)
    sweep_df = sweep_df.sort_values(["optimizer", "lr"], ascending=[True, False])
    sweep_df.to_csv(f"{OUT}/lr_sweep.csv", index=False)
    fig, ax = plt.subplots(figsize=(3.6, 2.9))
    for o in OPTIMIZERS:
        d = sweep_df[sweep_df.optimizer == o]
        ax.plot(d["lr"], d["test_acc"], "o-", label=o, color=COL[o])
    ax.set_xscale("log"); ax.set_xlabel("Learning rate"); ax.set_ylabel("Test accuracy (%)")
    ax.set_title("LR sensitivity"); ax.legend(frameon=False)
    plt.tight_layout()
    for ext in ("pdf", "png"): plt.savefig(f"{OUT}/figures/lr_sensitivity.{ext}", bbox_inches="tight")
    plt.show()
sweep_df[["optimizer", "lr", "test_acc", "best_val_acc", "train_time"]].round(2) if len(sweep_df) else None

## 10. Export LaTeX tables & macros (consumed by `report/` and `presentation/`)

In [ ]:
def fmt(m, s, d=2):  return f"{m:.{d}f} $\\pm$ {s:.{d}f}"
def na(v): return v is None or (isinstance(v, float) and math.isnan(v))

# ---- main results table
lines = [r"\begin{tabular}{lcccccc}", r"\toprule",
         r"Optimizer & Acc.\ (\%) & Prec.\ (\%) & Rec.\ (\%) & F1 (\%) & Time (s) & Ep.\ to " + f"{int(CFG['conv_thresh']*100)}" + r"\% val \\", r"\midrule"]
for o in OPTIMIZERS:
    r = summary.loc[o]
    ep = "--" if na(r["epochs_to_thr_mean"]) else fmt(r["epochs_to_thr_mean"], r["epochs_to_thr_std"], 1)
    cells = [o, fmt(r.test_acc_mean, r.test_acc_std), fmt(r.precision_mean, r.precision_std),
             fmt(r.recall_mean, r.recall_std), fmt(r.f1_mean, r.f1_std),
             fmt(r.train_time_mean, r.train_time_std, 0), ep]
    if o == best_opt: cells = [f"\\textbf{{{c}}}" for c in cells]
    lines.append(" & ".join(cells) + r" \\")
lines += [r"\bottomrule", r"\end{tabular}"]
open(f"{OUT}/results_table.tex", "w").write("\n".join(lines) + "\n")

# ---- LR sweep table
if len(sweep_df):
    L = [r"\begin{tabular}{llcc}", r"\toprule", r"Optimizer & LR & Test acc.\ (\%) & Best val acc.\ (\%) \\", r"\midrule"]
    for o in OPTIMIZERS:
        d = sweep_df[sweep_df.optimizer == o]
        for _, r in d.iterrows():
            a = f"{r.test_acc:.2f}"; b = f"{r.best_val_acc:.2f}"
            if r.test_acc == d.test_acc.max(): a = f"\\textbf{{{a}}}"
            L.append(f"{o if r.lr == d.lr.max() else ''} & {r.lr:g} & {a} & {b} \\\\")
        if o != list(OPTIMIZERS)[-1]: L.append(r"\midrule")
    L += [r"\bottomrule", r"\end{tabular}"]
    open(f"{OUT}/lr_sweep_table.tex", "w").write("\n".join(L) + "\n")

# ---- numeric macros
fastest = summary["sec_per_epoch_mean"].idxmin()
fastconv = summary["epochs_to_thr_mean"].idxmin() if summary["epochs_to_thr_mean"].notna().any() else "n/a"
M = []
def mac(n, v): M.append(f"\\newcommand{{\\{n}}}{{{v}}}")
mac("bestOpt", best_opt); mac("fastestOpt", fastest); mac("fastConvOpt", fastconv)
mac("nSeeds", len(CFG["seeds"])); mac("nEpochs", CFG["epochs"]); mac("sweepEpochs", CFG["sweep_epochs"])
mac("nParams", f"{N_PARAMS:,}".replace(",", "{,}")); mac("convThresh", int(CFG["conv_thresh"] * 100))
mac("bestAcc", f"{summary.loc[best_opt,'test_acc_mean']:.2f}"); mac("bestAccStd", f"{summary.loc[best_opt,'test_acc_std']:.2f}")
mac("worstOpt", summary["test_acc_mean"].idxmin()); mac("worstAcc", f"{summary['test_acc_mean'].min():.2f}")
mac("accGap", f"{summary['test_acc_mean'].max() - summary['test_acc_mean'].min():.2f}")
for o in OPTIMIZERS:
    r = summary.loc[o]
    mac(f"acc{o}", f"{r.test_acc_mean:.2f}"); mac(f"accstd{o}", f"{r.test_acc_std:.2f}")
    mac(f"prec{o}", f"{r.precision_mean:.2f}"); mac(f"rec{o}", f"{r.recall_mean:.2f}"); mac(f"fscore{o}", f"{r.f1_mean:.2f}")
    mac(f"time{o}", f"{r.train_time_mean:.0f}"); mac(f"spe{o}", f"{r.sec_per_epoch_mean:.1f}")
    mac(f"convep{o}", "--" if na(r.epochs_to_thr_mean) else f"{r.epochs_to_thr_mean:.1f}")
    mac(f"bestep{o}", f"{r.best_epoch_mean:.1f}")
open(f"{OUT}/numbers.tex", "w").write("\n".join(M) + "\n")

# ---- markdown table for the README
md_t = ["| Optimizer | Test Acc (%) | Precision (%) | Recall (%) | F1 (%) | Train time (s) |", "|---|---|---|---|---|---|"]
for o in OPTIMIZERS:
    r = summary.loc[o]
    md_t.append(f"| {o} | {r.test_acc_mean:.2f} ± {r.test_acc_std:.2f} | {r.precision_mean:.2f} | {r.recall_mean:.2f} | {r.f1_mean:.2f} | {r.train_time_mean:.0f} |")
open(f"{OUT}/results_table.md", "w").write("\n".join(md_t) + "\n")
print("\n".join(md_t)); print("\nBest optimizer:", best_opt)
print(open(f"{OUT}/numbers.tex").read()[:400])

## 11. Download everything
Copy the contents of `results/` into the repository's `results/` folder (overwriting the placeholders), then recompile `report/report.tex` and `presentation/presentation.tex`.

In [ ]:
import shutil
shutil.make_archive("results_bundle", "zip", OUT)
try:
    from google.colab import files
    files.download("results_bundle.zip")
except Exception as e:
    print("Zip saved as results_bundle.zip (not running on Colab):", e)